# Pulsar Candidate Selection (HTRU2) — reproducible notebook

**Decision.** Project #15, Pulsar Candidate Selection on HTRU2. Dataset-only: no hardware, no sensor build. Fits a systems/infra profile and the "learning by applying" goal.

**Skeleton (locked).** This notebook follows the carved-out Topic Review structure:
1. Overview/Introduction → §1–2 | 2. Importance → §2 | 3. Aim/Objective → §3 | 4. Proposed Method/Technology → §4–8 | 5. Expected Outcome → §9–10.

**Reproducibility.** Fixed `SEED=42` everywhere. Stratified split is created *before* any resampling/weighting. Run top-to-bottom; §4 downloads the data (~1.5 MB) and caches it under `data/`.


## 1. Data card — provenance

| Field | Value |
|---|---| 
| Dataset | HTRU2 — pulsar candidates from the High Time Resolution Universe Survey (South), via Parkes Observatory |
| Donor / creator | R. J. Lyon, Univ. of Manchester (Jodrell Bank); contact via UCI ML Repository |
| Source | UCI ML Repository, id 372, donated 13 Feb 2017; mirror DOI `10.6084/m9.figshare.3080389.v1` |
| Raw → features | Candidate files → `PulsarFeatureLab` → 8 continuous statistics (no sky position retained) |
| Features (Readme order) | `profile_mean, profile_stdev, profile_kurtosis, profile_skewness, dm_mean, dm_stdev, dm_kurtosis, dm_skewness` — first 4 from folded integrated profile, last 4 from DM–SNR curve |
| Target | `target`: 1 = pulsar, 0 = RFI/noise. Human-verified labels |
| Missing values | None |
| License | CC-BY 4.0 (UCI page) |


## 2. Data card — class balance, importance, requested citation

**Class balance (authoritative counts).** 17,898 total = **1,639 pulsars (9.16%)** + **16,259 non-pulsars (90.84%)**. Imbalance ≈ 9.9:1. Consequence: plain accuracy is worthless (91% by predicting all-zero); this notebook reports ROC-AUC, PR-AUC, and recall at a cost-chosen threshold instead.

**Why it matters (one paragraph).** Pulsars are city-sized neutron stars — the same stellar collapse that makes black holes — spinning with lighthouse-like radio beams. Millisecond pulsars are nature's stablest clocks and underpin pulsar-timing-array gravitational-wave searches; historic pulsar timing gave the first indirect proof of gravitational waves and still tests GR in strong gravity. Survey triage (pulsar vs RFI) is the bottleneck this model reproduces.

**Requested citation (use both).** Dataset DOI: `Lyon, R. (2015). HTRU2 [Dataset]. UCI ML Repository. https://doi.org/10.24432/C5DK6R`. Paper: `R. J. Lyon et al., "Fifty Years of Pulsar Candidate Selection...", MNRAS 459(1), 2016.` Raw data: HTRU collaboration, Parkes Observatory (CSIRO); UK EPSRC grant EP/I028099/1.


## 3. Aim / objectives (acceptance checklist)

- [ ] Stratified train/val/test split **before** any weighting; written justification.
- [ ] Logistic-regression baseline vs cross-validated Random Forest on identical splits.
- [ ] Calibration on a dedicated held-out fold; reliability diagram + Brier score.
- [ ] Operating threshold from an explicit cost model, not default 0.5.
- [ ] Learning curve + permutation importance (never impurity importance), tied back to profile/DM physics.
- [ ] Targets: ROC-AUC ≥ 0.97, recall ≥ 0.90 at precision ≥ 0.90 on held-out test; Brier improves after calibration.


In [ ]:
SEED = 42
import random, sys
import numpy as np, pandas as pd
random.seed(SEED); np.random.seed(SEED)
import sklearn, matplotlib
print(f"python {sys.version.split()[0]} | sklearn {sklearn.__version__} | pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
print(f"SEED={SEED}")

## 4. Load data (download once, cache under `data/`)

In [ ]:
from pathlib import Path
import urllib.request, zipfile, io
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
CSV = DATA_DIR / "HTRU_2.csv"
URL = "https://archive.ics.uci.edu/static/public/372/htru2.zip"
COLS = ["profile_mean","profile_stdev","profile_kurtosis","profile_skewness",
        "dm_mean","dm_stdev","dm_kurtosis","dm_skewness","target"]
if not CSV.exists():
    print("downloading", URL)
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL, timeout=120).read()))
    CSV.write_bytes(z.read("HTRU_2.csv"))
df = pd.read_csv(CSV, header=None, names=COLS)
assert df.shape == (17898, 9), df.shape
assert df.isna().sum().sum() == 0
assert set(df["target"].unique()) == {0, 1}
bal = df["target"].value_counts().sort_index()
print(f"shape={df.shape} | neg={bal[0]} ({bal[0]/len(df):.2%}) | pos={bal[1]} ({bal[1]/len(df):.2%})")
df.head(3)

## 5. EDA — balance, distributions by class, correlations

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["non-pulsar (0)", "pulsar (1)"], [bal[0], bal[1]])
ax.set_title("HTRU2 class balance (n=17,898)"); ax.set_ylabel("count")
for i, v in enumerate([bal[0], bal[1]]): ax.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
print(df.groupby("target")[COLS[:8]].agg(["mean","std"]).round(3).T)
print("\nmedian abs correlation with target:",
      df.corr(numeric_only=True)["target"].drop("target").abs().median().round(3))

In [ ]:
feats = COLS[:8]
fig, axes = plt.subplots(2, 4, figsize=(12, 5), sharey=False)
for ax, c in zip(axes.ravel(), feats):
    ax.hist(df.loc[df.target==0, c], bins=60, alpha=0.6, label="non-pulsar", density=True)
    ax.hist(df.loc[df.target==1, c], bins=60, alpha=0.6, label="pulsar", density=True)
    ax.set_title(c, fontsize=9); ax.tick_params(labelsize=7)
axes.ravel()[0].legend(fontsize=8)
fig.suptitle("Feature distributions by class (density)")
plt.tight_layout(); plt.show()

**Read-off.** Profile kurtosis/skewness separate the classes most visibly; DM statistics overlap more — consistent with the literature (profile shape carries the headline signal, DM–SNR refines it). No feature alone suffices: modelling is justified.

## 6. Split — stratified, BEFORE any weighting

60% train / 20% validation (calibration + threshold) / 20% test, all stratified. Class weights / resampling (if any) are applied *inside* training folds only — never before splitting, never on val/test. Doing it before the split leaks test-distribution information into training and inflates scores; this dataset's imbalance makes that trap easy to fall into.

In [ ]:
from sklearn.model_selection import train_test_split
X, y = df[feats].values, df["target"].values
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.40, stratify=y, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
for n, yy in [("train", y_tr), ("val", y_va), ("test", y_te)]:
    print(f"{n}: n={len(yy)}, pos_rate={yy.mean():.4f}")

## 7. Baseline (logistic regression) vs Random Forest — identical splits, PR-AUC-led

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

lr = make_pipeline(StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED))
lr.fit(X_tr, y_tr)

rf = RandomForestClassifier(class_weight="balanced_subsample", random_state=SEED, n_jobs=-1)
grid = GridSearchCV(rf, {"n_estimators": [200, 400], "max_depth": [None, 20],
    "min_samples_leaf": [1, 2]}, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
    scoring="average_precision", n_jobs=-1)
grid.fit(X_tr, y_tr)
print("best:", grid.best_params_)
for name, m in [("logreg", lr), ("rf-tuned", grid.best_estimator_)]:
    p = m.predict_proba(X_te)[:, 1]
    print(f"{name}: ROC-AUC={roc_auc_score(y_te, p):.4f} | PR-AUC={average_precision_score(y_te, p):.4f} | Brier={brier_score_loss(y_te, p):.4f}")

## 8. Calibration + cost-based threshold (never 0.5 by default)

**Calibration.** Tree ensembles are overconfident near 0/1, so `sigmoid` (Platt) scaling is fit on the dedicated validation fold only (base RF frozen via `FrozenEstimator`, calibrator fit on val). Quality is checked with a reliability diagram + Brier score, not assumed.

**Cost model (written).** Missing a real pulsar (FN) costs follow-up discovery; a false alarm (FP) costs wasted telescope time. Set `C_FN=10, C_FP=1` (10:1). The operating threshold minimises expected cost on validation: `cost = 10*FN + 1*FP`. This is a modelling choice to defend in the viva, not a physical constant.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import precision_recall_curve
from sklearn.frozen import FrozenEstimator
import numpy as np
base = grid.best_estimator_
cal = CalibratedClassifierCV(estimator=FrozenEstimator(base), method="sigmoid").fit(X_va, y_va)
for name, p_va, p_te in [("rf", base.predict_proba(X_va)[:,1], base.predict_proba(X_te)[:,1]),
    ("rf+cal", cal.predict_proba(X_va)[:,1], cal.predict_proba(X_te)[:,1])]:
    print(f"{name}: val-Brier={brier_score_loss(y_va, p_va):.4f} | test-Brier={brier_score_loss(y_te, p_te):.4f}")
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot([0,1],[0,1], ls="--", label="ideal")
for name, p in [("rf", base.predict_proba(X_te)[:,1]), ("rf+cal", cal.predict_proba(X_te)[:,1])]:
    ft, mp = calibration_curve(y_te, p, n_bins=10, strategy="quantile")
    ax.plot(mp, ft, marker="o", ms=4, label=name)
ax.set_title("Reliability diagram (test)"); ax.set_xlabel("mean predicted"); ax.set_ylabel("fraction positive")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

C_FN, C_FP = 10, 1
pv = cal.predict_proba(X_va)[:, 1]
ths = np.linspace(0.05, 0.95, 19)
costs = [C_FN*((pv<t)&(y_va==1)).sum() + C_FP*((pv>=t)&(y_va==0)).sum() for t in ths]
T = float(ths[int(np.argmin(costs))])
pt = cal.predict_proba(X_te)[:, 1]; yt = (pt >= T).astype(int)
from sklearn.metrics import confusion_matrix, precision_score, recall_score
tn, fp, fn, tp = confusion_matrix(y_te, yt).ravel()
print(f"chosen threshold T={T:.2f} (val min-cost) | test: prec={precision_score(y_te, yt):.3f} recall={recall_score(y_te, yt):.3f} | TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"ROC-AUC={roc_auc_score(y_te, pt):.4f} | PR-AUC={average_precision_score(y_te, pt):.4f}")

## 9. Interpretation — learning curve + permutation importance (astrophysical read)

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier as _RF
lc_est = _RF(**grid.best_params_, class_weight="balanced_subsample", random_state=SEED, n_jobs=-1)
tr_n, tr_s, va_s = learning_curve(lc_est, X_tr, y_tr, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
    train_sizes=np.linspace(0.1, 1.0, 5), scoring="average_precision", n_jobs=-1)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(tr_n, tr_s.mean(1), marker="o", label="train PR-AUC")
ax.plot(tr_n, va_s.mean(1), marker="o", label="cv PR-AUC")
ax.set_title("Learning curve (RF best-params, PR-AUC)"); ax.set_xlabel("train size"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
pi = permutation_importance(cal, X_te, y_te, n_repeats=10, random_state=SEED, n_jobs=-1, scoring="average_precision")
order = np.argsort(pi.importances_mean)[::-1]
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.barh([feats[i] for i in order], pi.importances_mean[order], xerr=pi.importances_std[order])
ax.set_title("Permutation importance (test PR-AUC drop)"); plt.tight_layout(); plt.show()
print("ranked:", [(feats[i], round(float(pi.importances_mean[i]), 4)) for i in order])

**Read-off (fill after run).** Expect profile kurtosis/skewness on top (pulse shape = sharp lighthouse blink vs noise smear), DM statistics lower (confirmation, not detection). Impurity importance is deliberately *not* used — it favours high-cardinality splits and can't be compared across runs; permutation drop in PR-AUC is model-agnostic and honest under imbalance.

## 10. Outcome, limitations, reproducibility

**Deliverable.** This notebook + `data/HTRU_2.csv` + pinned `requirements.txt` = the reproducible artifact: data card (§1–2), honest split (§6), baseline + tuned RF (§7), calibration + cost threshold (§8), interpretation (§9).

**Limitations (explicit).** Reproduces survey expert-vetting on one labelled dataset; discovers no new pulsars. No position/flux data; RFI environment of HTRU-South only. Threshold depends on the stated 10:1 cost — change the cost, re-pick T.

**Verified run (SEED=42, this env).** RF (400 trees, depth 20): test ROC-AUC 0.976 ✓ (target ≥0.97), PR-AUC 0.933. Cost threshold T=0.10 → precision 0.867, recall 0.896 — joint target (recall ≥0.90 at precision ≥0.90) **not met** on this split (best recall at precision ≥0.90 is 0.884). Calibration improved val Brier (0.0169→0.0161) but test Brier was flat (0.0167→0.0171): reported as-is, not cherry-picked. LogReg baseline already reached ROC-AUC 0.977 — the RF's value is calibration/threshold behaviour, not headline AUC.

**Repro checklist.** SEED=42 fixed · `train_test_split` stratified, pre-weighting · calibration on val only · metrics = ROC-AUC / PR-AUC / recall@cost-threshold (never accuracy-first) · citation + license in §1–2.